# 15. p01_squat_set1 One-Take Annotation + Report

This notebook prepares one real MediaPipe squat recording for a single-pass research review: import, data-quality screening, annotation authoring/checking, manual-rep preservation, pipeline execution, and report-table export.

- Stage title: Real-data import and report preparation / 실제 데이터 입력 및 리포트 준비
- Prerequisite: MediaPipe pose CSV already exported for `p01_squat_set1`
- Input: `data/pose/mediapipe/no_consent/20260517/p01_squat_set1_output_pose.csv`
- Optional input: `p01_squat_set1_annotation.csv` in the same folder
- Output target: `data/processed/reports/p01_squat_set1/`
- Validation point: annotation frame ranges must use the original `frame` numbers from the CSV/video, not row indices.

The notebook keeps the raw pose CSV unchanged. Empty leading frames and missing-frame rows are handled only in the working dataframe used by this notebook.

In [ ]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import Markdown, display

try:
    from scipy import signal as scipy_signal
except ImportError:  # pragma: no cover - notebook fallback
    scipy_signal = None

from movement.core.config import (
    CONNECTIONS,
    LANDMARKS,
    make_coordinate_columns,
    make_required_columns,
    make_visibility_columns,
)
from movement.core.io import load_pose_csv, print_data_summary
from movement.core.utils import compute_plot_ranges
from movement.definitions.exercise_definition import load_exercise_definition
from movement.pipeline import load_pipeline_config, run_pipeline
from movement.reporting.visualization import create_pose_animation
from movement.stages.annotation import (
    apply_annotation,
    load_annotation_csv,
    validate_annotation,
)
from movement.stages.recording_phase_split import (
    expected_phase_order_from_exercise,
    generate_recording_plane_phase_split,
    promote_phase_split_to_annotation,
    validate_phase_split_for_promotion,
)
from movement.stages.validation import run_basic_validation

print("imports OK")
print("project root:", PROJECT_ROOT)

## Input Settings

Change these variables first when reviewing another recording. For this p01 pass, keep frame labels in the original CSV coordinate system so the annotation CSV can be checked against the video and pose file directly.

In [ ]:
RECORDING_ID = "p01_squat_set1"
SESSION_ID = "20260517_no_consent_dev"

POSE_CSV = PROJECT_ROOT / "data/pose/mediapipe/no_consent/20260517/p01_squat_set1_output_pose.csv"
POSE_VIDEO = PROJECT_ROOT / "data/pose/mediapipe/no_consent/20260517/p01_squat_set1_output.mp4"
ANNOTATION_CSV = POSE_CSV.with_name("p01_squat_set1_annotation.csv")
OUTPUT_DIR = PROJECT_ROOT / "data/processed/reports/p01_squat_set1"
# Phase QC stays beside the annotation CSV because it is annotation-derived metadata.
PHASE_SPLIT_CSV = ANNOTATION_CSV.with_name(f"{RECORDING_ID}_phase_split.csv")
PHASE_ANNOTATION_CSV = ANNOTATION_CSV.with_name(f"{RECORDING_ID}_phase_annotation.csv")

DEFINITIONS_DIR = PROJECT_ROOT / "data/definitions/exercises"
CONFIG_PATH = PROJECT_ROOT / "configs/pipeline_default.yaml"
EXERCISE_ID = "squat"

# Squat camera protocol recommends Z2/Z8 and H2. The current p01 recording is Z8.
# Z8 is a front-left oblique view: frontal alignment and sagittal depth/ROM are both moderate-confidence families.
OBSERVED_CAMERA_ZONE = "Z8"
OBSERVED_CAMERA_HEIGHT_LEVEL = "H2"
REFERENCE_MAT_USED = False
FILMING_PROTOCOL_STATUS = "no_anchor"  # recommended | out_of_zone | no_anchor | unknown

SAVE_OUTPUTS = False
WRITE_ANNOTATION_CSV = False
WRITE_PHASE_SPLIT_CSV = True

# Promotion gate: keep both False until visual QC is complete.
PHASE_VISUAL_QC_CONFIRMED = False
PROMOTE_PHASE_SPLIT_TO_ANNOTATION = False

# Plotly 3D playback control. 1.00 means real-time speed; 0.50/2.00 work like video speed controls.
# PLAYBACK_STRIDE skips rendered frames but preserves real elapsed time, which helps browser-based 3D playback.
PLAYBACK_SPEED = 1.00
PLAYBACK_STRIDE = 2  # set to 1 for every frame if your browser can keep up

assert POSE_CSV.exists(), f"Missing pose CSV: {POSE_CSV}"
assert DEFINITIONS_DIR.exists(), f"Missing definitions dir: {DEFINITIONS_DIR}"
assert CONFIG_PATH.exists(), f"Missing pipeline config: {CONFIG_PATH}"

print("pose csv:", POSE_CSV.relative_to(PROJECT_ROOT))
print("annotation csv:", ANNOTATION_CSV.relative_to(PROJECT_ROOT))
print("video exists:", POSE_VIDEO.exists())

## Load And Regularize Pose CSV

MediaPipe exports can include leading frames before pose detection starts and occasional dropped frame rows. This cell removes fully empty leading/trailing pose rows, restores missing frame numbers as NaN rows, and rebuilds a monotonic timestamp for analysis. The original CSV is not changed.

In [ ]:
RAW_COORD_COLUMNS = [
    col
    for landmark in LANDMARKS
    for col in (f"{landmark}_x", f"{landmark}_y", f"{landmark}_z")
]
RAW_VISIBILITY_COLUMNS = [f"{landmark}_visibility" for landmark in LANDMARKS]


def estimate_fps_from_timestamp(df: pd.DataFrame, fallback: float = 30.0) -> float:
    if "timestamp" not in df.columns:
        return fallback
    dt = df["timestamp"].dropna().diff().dropna()
    dt = dt[dt > 0]
    if dt.empty:
        return fallback
    return float(1.0 / dt.median())


def trim_empty_pose_frames(df: pd.DataFrame) -> pd.DataFrame:
    present_cols = [col for col in RAW_COORD_COLUMNS if col in df.columns]
    nonempty = df[present_cols].notna().any(axis=1)
    if not nonempty.any():
        raise ValueError("No non-empty pose frames found.")
    return df.loc[nonempty].copy()


def regularize_frame_rows(df: pd.DataFrame, fps: float) -> pd.DataFrame:
    start_frame = int(df["frame"].min())
    end_frame = int(df["frame"].max())
    full_index = pd.DataFrame({"frame": range(start_frame, end_frame + 1)})
    payload = df.drop(columns=["timestamp"], errors="ignore")
    regularized = full_index.merge(payload, on="frame", how="left")
    regularized.insert(1, "timestamp", (regularized["frame"] - start_frame) / fps)
    return regularized


raw_df = load_pose_csv(POSE_CSV)
fps_estimate = estimate_fps_from_timestamp(raw_df)
trimmed_df = trim_empty_pose_frames(raw_df)
analysis_df = regularize_frame_rows(trimmed_df, fps=fps_estimate)
analysis_df.attrs["fps"] = fps_estimate

print_data_summary(raw_df)
summary = pd.DataFrame([
    {
        "raw_frames": len(raw_df),
        "analysis_frames": len(analysis_df),
        "raw_frame_min": int(raw_df["frame"].min()),
        "raw_frame_max": int(raw_df["frame"].max()),
        "analysis_frame_min": int(analysis_df["frame"].min()),
        "analysis_frame_max": int(analysis_df["frame"].max()),
        "trimmed_empty_frames": len(raw_df) - len(trimmed_df),
        "restored_missing_frame_rows": len(analysis_df) - len(trimmed_df),
        "fps_estimate": round(fps_estimate, 3),
    }
])
display(summary)

## Validation Gate

Structural checks should pass before annotation or visualization. Visibility can fail on real recordings; keep it as data-confidence provenance rather than a hard stop.

In [ ]:
validation_report = run_basic_validation(
    df=analysis_df,
    required_columns=make_required_columns(LANDMARKS),
    coordinate_columns=make_coordinate_columns(LANDMARKS),
    visibility_columns=make_visibility_columns(LANDMARKS),
)

structural_checks = [
    "required_columns",
    "frame_continuity",
    "timestamp",
    "missing_values",
]
structural_passed = all(validation_report[name]["passed"] for name in structural_checks)
visibility_passed = validation_report.get("visibility", {}).get("passed", True)

print("overall validation passed:", validation_report["passed"])
print("structural checks passed:", structural_passed)
print("visibility passed:", visibility_passed)

failed_sections = {
    name: section
    for name, section in validation_report.items()
    if isinstance(section, dict) and not section.get("passed", True)
}
print(json.dumps(failed_sections, indent=2, ensure_ascii=False)[:5000])

assert structural_passed, "Structural import checks failed; fix pose import before annotation."

## Exercise And Camera Context

Squat is bilateral. The current p01 recording is Z8, a front-left oblique view. This view supports a balanced but moderate-confidence read of frontal alignment and sagittal depth/ROM. View-sensitive features should carry reliability notes rather than being forced into a movement-quality penalty.

In [ ]:
exercise = load_exercise_definition(EXERCISE_ID, DEFINITIONS_DIR)
protocol = exercise.performance_protocol
camera_protocol = exercise.camera_protocol

context_rows = [
    {
        "exercise_id": exercise.exercise_id,
        "definition_version": exercise.version,
        "laterality": exercise.classification.get("laterality"),
        "primary_plane": exercise.classification.get("primary_plane"),
        "target_count_per_set": getattr(protocol.prescription, "target_count_per_set", None) if protocol else None,
        "count_unit": getattr(protocol.prescription, "count_unit", None) if protocol else None,
        "recommended_zones": ", ".join(camera_protocol.recommended_zones) if camera_protocol else None,
        "observed_zone": OBSERVED_CAMERA_ZONE,
        "recommended_height": camera_protocol.recommended_height if camera_protocol else None,
        "observed_height": OBSERVED_CAMERA_HEIGHT_LEVEL,
    }
]
display(pd.DataFrame(context_rows))

## Raw Trajectory Review For Annotation

Use this plot to write the annotation CSV. For MediaPipe image-style coordinates, larger `hip_center_y` usually means the pelvis is lower in the image. Candidate bottoms and tops are visual aids only; confirm rep boundaries against the video before using them.

In [ ]:
hip_center_y = (analysis_df["left_hip_y"] + analysis_df["right_hip_y"]) / 2.0
trace = hip_center_y.interpolate(method="linear").ffill().bfill()
smoothed_trace = trace.rolling(window=31, center=True, min_periods=1).median()

candidate_rows = []
if scipy_signal is not None:
    min_distance = max(8, int(round(fps_estimate * 1.0)))
    bottoms, _ = scipy_signal.find_peaks(
        smoothed_trace.to_numpy(), distance=min_distance, prominence=0.05
    )
    tops, _ = scipy_signal.find_peaks(
        (-smoothed_trace).to_numpy(), distance=min_distance, prominence=0.05
    )
    for local_idx in bottoms:
        candidate_rows.append(
            {
                "marker_type": "candidate_bottom",
                "frame": int(analysis_df.iloc[int(local_idx)]["frame"]),
                "hip_center_y": float(hip_center_y.iloc[int(local_idx)]),
            }
        )
    for local_idx in tops:
        candidate_rows.append(
            {
                "marker_type": "candidate_top",
                "frame": int(analysis_df.iloc[int(local_idx)]["frame"]),
                "hip_center_y": float(hip_center_y.iloc[int(local_idx)]),
            }
        )

candidate_table = pd.DataFrame(candidate_rows).sort_values("frame").reset_index(drop=True)
display(candidate_table)

fig = go.Figure()
fig.add_scatter(
    x=analysis_df["frame"],
    y=hip_center_y,
    mode="lines",
    name="hip_center_y_raw",
    line=dict(color="#7a7a7a", width=1),
)
fig.add_scatter(
    x=analysis_df["frame"],
    y=smoothed_trace,
    mode="lines",
    name="hip_center_y_smoothed",
    line=dict(color="#1f77b4", width=2),
)
if not candidate_table.empty:
    for marker_type, color, symbol in [
        ("candidate_bottom", "#d62728", "triangle-down"),
        ("candidate_top", "#2ca02c", "triangle-up"),
    ]:
        subset = candidate_table[candidate_table["marker_type"] == marker_type]
        fig.add_scatter(
            x=subset["frame"],
            y=subset["hip_center_y"],
            mode="markers",
            name=marker_type,
            marker=dict(color=color, symbol=symbol, size=10),
        )
fig.update_layout(
    title="p01_squat_set1 hip-center trajectory for annotation",
    xaxis_title="original frame number",
    yaxis_title="hip_center_y (raw MediaPipe coordinate)",
    height=460,
)
fig.show()

## Annotation CSV 작성법

필수 컬럼은 아래 6개입니다.

```csv
segment_type,set_id,rep_id,start_frame,end_frame,use_for_analysis
```

p01 리포트용으로는 다음 optional provenance 컬럼까지 함께 쓰는 것을 권장합니다.

```csv
exercise_type,pattern,session_id,recording_id,set_index,camera_zone,camera_height_level,
reference_mat_used,filming_protocol_status,performance_protocol_status,actual_rep_count,
failure_point_frame,failure_rep_id,failure_reason,performance_note,rep_unit,note
```

작성 규칙:

- `start_frame`, `end_frame`은 원본 CSV/video의 `frame` 번호이며 inclusive range입니다.
- row끼리 frame range가 겹치면 안 됩니다.
- `rep` row는 complete descent + ascent 한 사이클을 담습니다.
- 첫 pass에서는 `rep` row만 작성해도 됩니다. annotation이 있으면 annotation 밖의 프레임은 자동 제외됩니다.
- baseline/idle/transition은 리포트 provenance에는 좋지만, 분석에는 `use_for_analysis=false`로 둡니다.
- squat은 `exercise_type=squat`, `pattern=bilateral`, `rep_unit=repetition`을 사용합니다.
- `phase`는 선택입니다. 현재 p01에서는 rep annotation을 먼저 확정하고, 다음 섹션에서 recording-plane 반자동 phase sample을 생성해 시각적으로 확인합니다.
- 실패/중단이 없으면 `performance_protocol_status=completed`, `failure_*`는 비워둡니다.
- 실패/중단이 있으면 `performance_protocol_status=partial` 또는 `stopped_at_failure_point`로 쓰고 `failure_point_frame`, `failure_rep_id`, `failure_reason`을 기록합니다.

예시 형식:

```csv
segment_type,set_id,rep_id,start_frame,end_frame,use_for_analysis,exercise_type,pattern,session_id,recording_id,set_index,camera_zone,camera_height_level,reference_mat_used,filming_protocol_status,performance_protocol_status,actual_rep_count,failure_point_frame,failure_rep_id,failure_reason,performance_note,rep_unit,note
rep,1,1,90,175,true,squat,bilateral,20260517_no_consent_dev,p01_squat_set1,1,Z8,H2,false,no_anchor,completed,10,,,,,repetition,complete squat rep 1
rep,1,2,176,237,true,squat,bilateral,20260517_no_consent_dev,p01_squat_set1,1,Z8,H2,false,no_anchor,completed,10,,,,,repetition,complete squat rep 2
```

Save the file as:

`data/pose/mediapipe/no_consent/20260517/p01_squat_set1_annotation.csv`

In [ ]:
ANNOTATION_COLUMNS = [
    "segment_type",
    "set_id",
    "rep_id",
    "start_frame",
    "end_frame",
    "use_for_analysis",
    "exercise_type",
    "pattern",
    "session_id",
    "recording_id",
    "set_index",
    "camera_zone",
    "camera_height_level",
    "reference_mat_used",
    "filming_protocol_status",
    "performance_protocol_status",
    "actual_rep_count",
    "failure_point_frame",
    "failure_rep_id",
    "failure_reason",
    "performance_note",
    "rep_unit",
    "note",
]

empty_annotation_template = pd.DataFrame(columns=ANNOTATION_COLUMNS)
display(empty_annotation_template)

## Build Or Load Annotation

Option A: create `p01_squat_set1_annotation.csv` in the same folder as the pose CSV and rerun this cell.

Option B: fill `MANUAL_REP_RANGES` below, set `WRITE_ANNOTATION_CSV=True`, and rerun the cell to write a draft CSV. Keep the ranges non-overlapping and use original frame numbers.

In [ ]:
# Fill this list after reviewing the trajectory plot and video.
# Format: (rep_id, start_frame, end_frame)
MANUAL_REP_RANGES = [
    # (1, 90, 175),
    # (2, 176, 237),
]

# Optional non-analysis rows. Format: (segment_type, start_frame, end_frame, note)
MANUAL_NON_ANALYSIS_RANGES = [
    # ("baseline", 18, 89, "standing setup before first analyzed rep"),
    # ("idle", 852, 883, "after final analyzed rep"),
]

ANNOTATION_METADATA = {
    "exercise_type": EXERCISE_ID,
    "pattern": "bilateral",
    "session_id": SESSION_ID,
    "recording_id": RECORDING_ID,
    "set_index": 1,
    "camera_zone": OBSERVED_CAMERA_ZONE,
    "camera_height_level": OBSERVED_CAMERA_HEIGHT_LEVEL,
    "reference_mat_used": REFERENCE_MAT_USED,
    "filming_protocol_status": FILMING_PROTOCOL_STATUS,
    "performance_protocol_status": "completed",
    "actual_rep_count": len(MANUAL_REP_RANGES) if MANUAL_REP_RANGES else pd.NA,
    "failure_point_frame": pd.NA,
    "failure_rep_id": pd.NA,
    "failure_reason": pd.NA,
    "performance_note": pd.NA,
    "rep_unit": "repetition",
}


def make_annotation_from_manual_ranges() -> pd.DataFrame:
    rows = []
    for segment_type, start_frame, end_frame, note in MANUAL_NON_ANALYSIS_RANGES:
        rows.append(
            {
                "segment_type": segment_type,
                "set_id": 1 if segment_type in {"rest", "transition"} else pd.NA,
                "rep_id": pd.NA,
                "start_frame": int(start_frame),
                "end_frame": int(end_frame),
                "use_for_analysis": False,
                **ANNOTATION_METADATA,
                "note": note,
            }
        )
    for rep_id, start_frame, end_frame in MANUAL_REP_RANGES:
        rows.append(
            {
                "segment_type": "rep",
                "set_id": 1,
                "rep_id": int(rep_id),
                "start_frame": int(start_frame),
                "end_frame": int(end_frame),
                "use_for_analysis": True,
                **ANNOTATION_METADATA,
                "note": f"complete squat rep {int(rep_id)}",
            }
        )
    if not rows:
        return pd.DataFrame(columns=ANNOTATION_COLUMNS)
    return pd.DataFrame(rows, columns=ANNOTATION_COLUMNS).sort_values("start_frame")


annotation_source = None
ann_df = None

if ANNOTATION_CSV.exists():
    ann_df = load_annotation_csv(ANNOTATION_CSV)
    annotation_source = "csv"
elif MANUAL_REP_RANGES:
    ann_df = make_annotation_from_manual_ranges()
    annotation_source = "manual_ranges"
    if WRITE_ANNOTATION_CSV:
        ANNOTATION_CSV.parent.mkdir(parents=True, exist_ok=True)
        ann_df.to_csv(ANNOTATION_CSV, index=False)
        annotation_source = "manual_ranges_written_to_csv"
else:
    print("No annotation CSV found and MANUAL_REP_RANGES is empty.")
    print("Write the annotation file or fill MANUAL_REP_RANGES, then rerun from this cell.")
    display(empty_annotation_template)

if ann_df is not None:
    display(Markdown(f"**Annotation source:** `{annotation_source}`"))
    display(ann_df)
    annotation_validation = validate_annotation(ann_df, analysis_df)
    print(json.dumps(annotation_validation, indent=2, ensure_ascii=False))
    assert annotation_validation["passed"], "Annotation validation failed."

## Annotation Application Check

When annotation is supplied, frames outside annotated ranges are excluded. For the first p01 report, this is intentional: the final report should be based on complete, reviewed squat reps only.

In [ ]:
annotated_df = None
annotation_report = None

if ann_df is None:
    print("Annotation not ready. Pipeline report cells will stay skipped.")
else:
    annotated_df, annotation_report = apply_annotation(analysis_df, ann_df)
    display(pd.DataFrame([
        {
            "annotation_provided": annotation_report["annotation_provided"],
            "num_total_frames": annotation_report["num_total_frames"],
            "num_analysis_frames": annotation_report["num_analysis_frames"],
            "num_excluded_frames": annotation_report["num_excluded_frames"],
            "num_reps": annotation_report["num_reps"],
            "performance_available": annotation_report["performance_provenance"]["available"],
            "performance_policy": annotation_report["performance_provenance"]["policy"],
        }
    ]))
    display(
        annotated_df[
            ["frame", "timestamp", "use_for_analysis", "segment_type", "set_id", "rep_id", "camera_zone"]
        ].head(12)
    )

## Camera-Space Recording View

This view keeps a fixed camera and fixed axis scale for visual QC. Playback speed is controlled by `PLAYBACK_SPEED` in the input settings. `1.00` requests real-time speed from the pose timestamps; values such as `0.50` and `2.00` behave like video playback multipliers. `PLAYBACK_STRIDE` can skip rendered frames while preserving elapsed time when Plotly 3D rendering is slower than real video playback.

In [ ]:
def equalize_plot_ranges(ranges: tuple[list[float], list[float], list[float]]):
    centers = [sum(axis_range) / 2.0 for axis_range in ranges]
    spans = [max(axis_range[1] - axis_range[0], 1e-9) for axis_range in ranges]
    half_span = max(spans) / 2.0
    return [[center - half_span, center + half_span] for center in centers]


GLOBAL_RAW_RANGES = equalize_plot_ranges(
    compute_plot_ranges(
        df=analysis_df,
        landmarks=LANDMARKS,
        padding=0.05,
        coord_mode="raw",
    )
)

# recording: camera-like QC view. oblique: depth-proxy inspection view.
RENDER_VIEW_MODE = "recording"  # recording | oblique
CAMERA_VIEW_PRESETS = {
    "recording": dict(
        eye=dict(x=0.0, y=-2.5, z=0.02),
        up=dict(x=0.0, y=0.0, z=1.0),
        center=dict(x=0.0, y=0.0, z=0.0),
        projection=dict(type="orthographic"),
    ),
    "oblique": dict(
        eye=dict(x=1.35, y=-1.9, z=0.9),
        up=dict(x=0.0, y=0.0, z=1.0),
        center=dict(x=0.0, y=0.0, z=0.0),
        projection=dict(type="orthographic"),
    ),
}


def apply_camera_space_scene(fig, *, title: str | None = None, view_mode: str = RENDER_VIEW_MODE):
    if view_mode not in CAMERA_VIEW_PRESETS:
        raise ValueError(f"Unknown view_mode: {view_mode}")
    x_range, y_range, z_range = GLOBAL_RAW_RANGES
    camera_preset = CAMERA_VIEW_PRESETS[view_mode]
    fig.update_layout(
        title=title or fig.layout.title.text,
        scene=dict(
            xaxis=dict(title="Image X / left-right", range=x_range, autorange=False),
            yaxis=dict(title="MediaPipe Z / depth proxy", range=y_range, autorange=False),
            zaxis=dict(title="-Image Y / apparent height", range=z_range, autorange=False),
            aspectmode="manual",
            aspectratio=dict(x=1, y=1, z=1),
            camera=camera_preset,
        ),
        uirevision=f"p01_squat_fixed_camera_space_{view_mode}",
    )
    return fig



def format_frame_status(row: pd.Series) -> str:
    segment = row.get("segment_type")
    if pd.isna(segment):
        segment = "excluded"
    segment = str(segment)

    frame = int(row["frame"]) if "frame" in row and pd.notna(row["frame"]) else None
    rep_id = row.get("rep_id")
    phase = row.get("recording_plane_phase")
    if pd.isna(phase):
        phase = row.get("phase")

    if segment == "rep" and pd.notna(rep_id):
        status = f"rep {int(rep_id)}"
        if pd.notna(phase):
            status = f"{status} ({phase})"
    elif segment in {"baseline", "idle", "transition", "rest", "excluded", "full_sequence"}:
        status = segment
    else:
        status = segment or "excluded"

    return f"frame {frame} | {status}" if frame is not None else status


def _status_box_annotation(status_text: str) -> dict:
    return dict(
        text=f"<b>{status_text}</b>",
        x=0.015,
        y=0.985,
        xref="paper",
        yref="paper",
        xanchor="left",
        yanchor="top",
        align="left",
        showarrow=False,
        bgcolor="rgba(255,255,255,0.88)",
        bordercolor="rgba(40,40,40,0.55)",
        borderwidth=1,
        borderpad=6,
        font=dict(size=14, color="#1f1f1f"),
    )


def add_frame_status_box(fig, frame_df: pd.DataFrame):
    """Attach a dynamic status text box to a Plotly pose animation."""
    if frame_df.empty or not fig.frames:
        return fig

    status_texts = [format_frame_status(row) for _, row in frame_df.reset_index(drop=True).iterrows()]
    fig.update_layout(annotations=[_status_box_annotation(status_texts[0])])

    updated_frames = []
    for i, frame_obj in enumerate(fig.frames):
        status = status_texts[min(i, len(status_texts) - 1)]
        frame_obj.layout = {"annotations": [_status_box_annotation(status)]}
        updated_frames.append(frame_obj)
    fig.frames = updated_frames

    # Show original source frame numbers in the slider labels when available.
    if fig.layout.sliders and "frame" in frame_df.columns:
        frame_numbers = frame_df["frame"].astype(int).reset_index(drop=True).tolist()
        slider = fig.layout.sliders[0]
        slider.currentvalue = {"prefix": "Pose frame: "}
        for i, step in enumerate(slider.steps):
            if i < len(frame_numbers):
                step.label = str(frame_numbers[i])

    return fig



def playback_frame_duration_ms(
    fps: float,
    *,
    stride: int = PLAYBACK_STRIDE,
    speed: float = PLAYBACK_SPEED,
) -> int:
    """Return Plotly frame duration that preserves source elapsed time."""
    fps = max(float(fps), 1e-6)
    stride = max(1, int(stride))
    speed = max(float(speed), 1e-6)
    return max(1, int(round(1000.0 * stride / fps / speed)))


def make_playback_df(df: pd.DataFrame, *, stride: int = PLAYBACK_STRIDE) -> pd.DataFrame:
    """Subsample frames for browser playback while keeping the final frame."""
    if df is None or df.empty:
        return df.copy() if df is not None else pd.DataFrame()
    stride = max(1, int(stride))
    if stride == 1:
        return df.copy()
    playback_df = df.iloc[::stride].copy()
    if "frame" in df.columns and int(playback_df["frame"].iloc[-1]) != int(df["frame"].iloc[-1]):
        playback_df = pd.concat([playback_df, df.tail(1)], axis=0)
    return playback_df.reset_index(drop=True)


PLAYBACK_FRAME_DURATION_MS = playback_frame_duration_ms(fps_estimate)
display(
    pd.DataFrame(
        [
            {
                "fps_estimate": round(float(fps_estimate), 3),
                "playback_speed_x": float(PLAYBACK_SPEED),
                "playback_stride": int(PLAYBACK_STRIDE),
                "requested_plotly_frame_duration_ms": PLAYBACK_FRAME_DURATION_MS,
                "rendered_fps_request": round(1000.0 / PLAYBACK_FRAME_DURATION_MS, 2),
                "source_time_preserved": True,
            }
        ]
    )
)


analysis_playback_df = make_playback_df(analysis_df)

fig_camera_space = create_pose_animation(
    df=analysis_playback_df,
    landmarks=LANDMARKS,
    connections=CONNECTIONS,
    coord_mode="raw",
    frame_duration=PLAYBACK_FRAME_DURATION_MS,
    height=760,
    width=1050,
    title=f"p01_squat_set1 raw camera-space review - {RENDER_VIEW_MODE} fixed scale ({PLAYBACK_SPEED:.2f}x)",
    show_text=False,
)
apply_camera_space_scene(fig_camera_space)
fig_camera_space.show()

## Rep Annotation Visual QC

Use this section to verify whether each annotated rep starts and ends at a sensible point. The colored bands are the annotation ranges, and the hip-center trace is a visual reference for squat depth. If a band starts too early or ends too late, edit `start_frame` / `end_frame` in the annotation CSV and rerun from `Build Or Load Annotation`.

In [ ]:
if ann_df is None or annotated_df is None:
    print("Skipped: annotation is not ready.")
else:
    rep_rows = ann_df[ann_df["segment_type"].eq("rep")].copy()
    rep_rows["duration_s"] = (rep_rows["end_frame"] - rep_rows["start_frame"] + 1) / fps_estimate

    qc_rows = []
    for _, row in rep_rows.iterrows():
        rep_id = int(row["rep_id"])
        start = int(row["start_frame"])
        end = int(row["end_frame"])
        rep_mask = annotated_df["frame"].between(start, end)
        rep_trace = hip_center_y.loc[rep_mask]
        if rep_trace.dropna().empty:
            bottom_frame = pd.NA
            bottom_y = np.nan
        else:
            bottom_idx = rep_trace.idxmax()
            bottom_frame = int(annotated_df.loc[bottom_idx, "frame"])
            bottom_y = float(rep_trace.loc[bottom_idx])
        qc_rows.append(
            {
                "rep_id": rep_id,
                "start_frame": start,
                "bottom_frame_estimate": bottom_frame,
                "end_frame": end,
                "duration_s": round(float(row["duration_s"]), 2),
                "bottom_hip_center_y": round(bottom_y, 4) if np.isfinite(bottom_y) else np.nan,
                "note": row.get("note", ""),
            }
        )

    rep_qc_df = pd.DataFrame(qc_rows)
    display(rep_qc_df)

    fig_qc = go.Figure()
    fig_qc.add_scatter(
        x=analysis_df["frame"],
        y=hip_center_y,
        mode="lines",
        name="hip_center_y_raw",
        line=dict(color="#9a9a9a", width=1),
    )
    fig_qc.add_scatter(
        x=analysis_df["frame"],
        y=smoothed_trace,
        mode="lines",
        name="hip_center_y_smoothed",
        line=dict(color="#1f77b4", width=2),
    )

    band_colors = ["rgba(31,119,180,0.16)", "rgba(44,160,44,0.14)"]
    for i, row in rep_rows.reset_index(drop=True).iterrows():
        start = int(row["start_frame"])
        end = int(row["end_frame"])
        rep_id = int(row["rep_id"])
        fig_qc.add_vrect(
            x0=start,
            x1=end,
            fillcolor=band_colors[i % 2],
            line_width=0,
            layer="below",
            annotation_text=f"rep {rep_id}",
            annotation_position="top left",
        )
    if not rep_qc_df.empty:
        fig_qc.add_scatter(
            x=rep_qc_df["bottom_frame_estimate"],
            y=rep_qc_df["bottom_hip_center_y"],
            mode="markers+text",
            text=[f"b{int(rep_id)}" for rep_id in rep_qc_df["rep_id"]],
            textposition="bottom center",
            name="estimated bottom",
            marker=dict(color="#d62728", size=9, symbol="triangle-down"),
        )
    fig_qc.update_layout(
        title="Annotation bands over hip-center trajectory",
        xaxis_title="original frame number",
        yaxis_title="hip_center_y (larger usually means lower pelvis in image)",
        height=520,
        width=1050,
    )
    fig_qc.show()

## Recording-Plane Phase Split QC

This section generates a semi-automatic phase split from the manually confirmed rep ranges. It uses the recording-plane `hip_center_y` trace because MediaPipe `z` is a depth proxy, not vertical height. Review the bottom markers and phase bands visually; if a bottom frame is off, add that rep to `MANUAL_BOTTOM_FRAME_OVERRIDES` and rerun this cell. The output is written to `p01_squat_set1_phase_split.csv` beside the annotation file.

In [ ]:
PHASE_SMOOTH_WINDOW_FRAMES = 9
PHASE_HOLD_HALF_WINDOW_FRAMES = max(2, int(round(fps_estimate * 0.10)))

# Optional manual correction after visual QC. Format: rep_id: bottom_frame
MANUAL_BOTTOM_FRAME_OVERRIDES = {
    # 1: 143,
}

phase_split_df = pd.DataFrame()
phase_qc_df = pd.DataFrame()
phase_labeled_df = None

if ann_df is None or annotated_df is None:
    print("Skipped: annotation is not ready.")
else:
    phase_split_df, phase_qc_df = generate_recording_plane_phase_split(
        analysis_df,
        ann_df,
        recording_id=RECORDING_ID,
        camera_zone=OBSERVED_CAMERA_ZONE,
        smooth_window_frames=PHASE_SMOOTH_WINDOW_FRAMES,
        hold_half_window_frames=PHASE_HOLD_HALF_WINDOW_FRAMES,
        manual_bottom_frame_overrides=MANUAL_BOTTOM_FRAME_OVERRIDES,
    )

    phase_labeled_df = annotated_df.copy()
    phase_labeled_df["recording_plane_phase"] = pd.NA
    for _, phase_row in phase_split_df.iterrows():
        mask = phase_labeled_df["frame"].between(
            int(phase_row["start_frame"]),
            int(phase_row["end_frame"]),
        )
        phase_labeled_df.loc[mask, "recording_plane_phase"] = phase_row["phase"]

    display(Markdown("**Semi-automatic phase QC summary**"))
    display(phase_qc_df)
    display(Markdown("**Phase range sample**"))
    display(phase_split_df)

    if WRITE_PHASE_SPLIT_CSV and not phase_split_df.empty:
        PHASE_SPLIT_CSV.parent.mkdir(parents=True, exist_ok=True)
        phase_split_df.to_csv(PHASE_SPLIT_CSV, index=False)
        print("wrote:", PHASE_SPLIT_CSV.relative_to(PROJECT_ROOT))

    trace_by_frame = pd.Series(
        hip_center_y.to_numpy(dtype=float),
        index=analysis_df["frame"].astype(int),
        name="hip_center_y",
    )

    fig_phase = go.Figure()
    fig_phase.add_scatter(
        x=analysis_df["frame"],
        y=hip_center_y,
        mode="lines",
        name="hip_center_y_raw",
        line=dict(color="#9a9a9a", width=1),
    )
    fig_phase.add_scatter(
        x=analysis_df["frame"],
        y=smoothed_trace,
        mode="lines",
        name="hip_center_y_smoothed_global",
        line=dict(color="#1f77b4", width=2),
    )

    phase_colors = {
        "Descent": "rgba(31,119,180,0.13)",
        "Turnaround_Hold": "rgba(214,39,40,0.22)",
        "Ascent": "rgba(44,160,44,0.13)",
    }
    for _, phase_row in phase_split_df.iterrows():
        fig_phase.add_vrect(
            x0=int(phase_row["start_frame"]),
            x1=int(phase_row["end_frame"]),
            fillcolor=phase_colors.get(str(phase_row["phase"]), "rgba(0,0,0,0.08)"),
            line_width=0,
            layer="below",
        )

    if not phase_qc_df.empty:
        bottoms = phase_qc_df.dropna(subset=["bottom_frame_estimate"]).copy()
        bottom_y = [trace_by_frame.get(int(frame), np.nan) for frame in bottoms["bottom_frame_estimate"]]
        fig_phase.add_scatter(
            x=bottoms["bottom_frame_estimate"],
            y=bottom_y,
            mode="markers+text",
            text=[f"b{int(rep_id)}" for rep_id in bottoms["rep_id"]],
            textposition="bottom center",
            name="estimated bottom",
            marker=dict(color="#d62728", size=9, symbol="triangle-down"),
        )

    fig_phase.update_layout(
        title="Recording-plane semi-automatic phase split QC",
        xaxis_title="original frame number",
        yaxis_title="hip_center_y (larger usually means lower pelvis in image)",
        height=540,
        width=1050,
    )
    fig_phase.show()


## Promote Phase Split To Phase Annotation

Use this only after visual QC. The promotion gate validates that the phase split covers every annotated rep exactly, follows the squat phase order, keeps the bottom frame inside `Turnaround_Hold`, and preserves filming provenance. Set `PHASE_VISUAL_QC_CONFIRMED=True` and `PROMOTE_PHASE_SPLIT_TO_ANNOTATION=True` in the input settings, then rerun this cell to write `p01_squat_set1_phase_annotation.csv`.

In [ ]:
phase_candidate_df = globals().get("phase_split_df", pd.DataFrame())
if (phase_candidate_df is None or phase_candidate_df.empty) and PHASE_SPLIT_CSV.exists():
    phase_candidate_df = pd.read_csv(PHASE_SPLIT_CSV)

phase_annotation_df = pd.DataFrame()
phase_promotion_report = None
phase_promotion_detail_df = pd.DataFrame()

if ann_df is None:
    print("Skipped: annotation is required before promotion.")
elif phase_candidate_df is None or phase_candidate_df.empty:
    print("Skipped: phase split is empty. Run Recording-Plane Phase Split QC first.")
else:
    expected_order = expected_phase_order_from_exercise(exercise)
    phase_promotion_report, phase_promotion_detail_df = validate_phase_split_for_promotion(
        phase_candidate_df,
        ann_df,
        expected_phase_order=expected_order,
        expected_camera_zone=OBSERVED_CAMERA_ZONE,
    )
    display(Markdown("**Phase promotion validation**"))
    display(pd.DataFrame([phase_promotion_report]))
    display(phase_promotion_detail_df)

    if PROMOTE_PHASE_SPLIT_TO_ANNOTATION:
        phase_annotation_df, phase_promotion_report, phase_promotion_detail_df = promote_phase_split_to_annotation(
            phase_candidate_df,
            ann_df,
            visual_qc_confirmed=PHASE_VISUAL_QC_CONFIRMED,
            expected_phase_order=expected_order,
            expected_camera_zone=OBSERVED_CAMERA_ZONE,
            source_phase_split_file=PHASE_SPLIT_CSV,
        )
        phase_annotation_df.to_csv(PHASE_ANNOTATION_CSV, index=False)
        print("wrote:", PHASE_ANNOTATION_CSV.relative_to(PROJECT_ROOT))
    else:
        print("Not promoted. Set PHASE_VISUAL_QC_CONFIRMED=True and PROMOTE_PHASE_SPLIT_TO_ANNOTATION=True after QC.")


## Single-Rep Recording View Preview

Change `REVIEW_REP_ID` and rerun this cell to inspect one annotated repetition at a time in the same fixed recording-style raw scene used above. Use this together with the phase QC plot: if the estimated bottom marker is off in the animation, add a `MANUAL_BOTTOM_FRAME_OVERRIDES` entry and rerun the phase QC cell.

In [ ]:
REVIEW_REP_ID = 1

if ann_df is None or annotated_df is None:
    print("Skipped: annotation is not ready.")
else:
    phase_preview_df = globals().get("phase_labeled_df")
    preview_df = phase_preview_df if phase_preview_df is not None else annotated_df
    rep_mask = preview_df["segment_type"].eq("rep") & preview_df["rep_id"].eq(REVIEW_REP_ID)
    rep_df = preview_df.loc[rep_mask].copy()
    if rep_df.empty:
        print(f"No frames found for rep_id={REVIEW_REP_ID}.")
    else:
        start = int(rep_df["frame"].min())
        end = int(rep_df["frame"].max())
        phase_counts = (
            rep_df["recording_plane_phase"].value_counts(dropna=False).to_dict()
            if "recording_plane_phase" in rep_df.columns
            else {}
        )
        display(
            pd.DataFrame(
                [
                    {
                        "rep_id": REVIEW_REP_ID,
                        "start_frame": start,
                        "end_frame": end,
                        "num_frames": len(rep_df),
                        "playback_frames": len(make_playback_df(rep_df)),
                        "duration_s": round(len(rep_df) / fps_estimate, 2),
                        "playback_speed_x": float(PLAYBACK_SPEED),
                        "playback_stride": int(PLAYBACK_STRIDE),
                        "phase_source": "recording_plane_phase" if "recording_plane_phase" in rep_df.columns else "none",
                        "phase_counts": phase_counts,
                    }
                ]
            )
        )
        rep_playback_df = make_playback_df(rep_df)
        fig_rep = create_pose_animation(
            df=rep_playback_df,
            landmarks=LANDMARKS,
            connections=CONNECTIONS,
            coord_mode="raw",
            frame_duration=PLAYBACK_FRAME_DURATION_MS,
            height=760,
            width=1050,
            title=f"p01_squat_set1 rep {REVIEW_REP_ID} raw recording-view preview ({start}-{end}, {PLAYBACK_SPEED:.2f}x)",
            show_text=False,
        )
        apply_camera_space_scene(fig_rep)
        add_frame_status_box(fig_rep, rep_playback_df)
        fig_rep.show()


## Full Raw Pose Animation

This repeats the same fixed recording-view scene over the full analyzed recording. It is redundant with the camera-space review above, but useful after editing annotation because it sits near the pipeline run cells.

In [ ]:
full_anim_df = make_playback_df(analysis_df)

fig_raw = create_pose_animation(
    df=full_anim_df,
    landmarks=LANDMARKS,
    connections=CONNECTIONS,
    coord_mode="raw",
    frame_duration=PLAYBACK_FRAME_DURATION_MS,
    height=760,
    width=1050,
    title=f"p01_squat_set1 raw pose review - {RENDER_VIEW_MODE} fixed scale ({PLAYBACK_SPEED:.2f}x)",
    show_text=False,
)
apply_camera_space_scene(fig_raw)
phase_preview_df = globals().get("phase_labeled_df")
full_preview_df = phase_preview_df if phase_preview_df is not None else annotated_df
if full_preview_df is not None:
    add_frame_status_box(fig_raw, make_playback_df(full_preview_df))
fig_raw.show()

## One-Take Pipeline Run

This run preserves manual `rep_id` labels from annotation. The built-in generic phase segmentation remains disabled for this p01 report because its current vertical-axis assumption uses normalized `z`, while this MediaPipe recording needs recording-plane phase QC from `hip_center_y`. Rep-level feature, biomech, biomarker, and provenance outputs still run; the semi-automatic phase sample above is the current phase-review artifact.

In [ ]:
analysis_result_df = None
analysis_report = None

if ann_df is None:
    print("Skipped: annotation is required for rep-level one-take analysis.")
else:
    cfg = load_pipeline_config(CONFIG_PATH)
    cfg.annotation.enabled = True
    cfg.exercise_definition.enabled = True
    cfg.exercise_definition.exercise_id = EXERCISE_ID
    cfg.preprocessing.enabled = True
    cfg.preprocessing.interpolation.enabled = True
    cfg.preprocessing.far_side_stabilization.enabled = True
    cfg.preprocessing.far_side_stabilization.jitter_threshold_torso_per_sec = 1.0
    cfg.preprocessing.far_side_stabilization.acceleration_threshold_torso_per_sec2 = 30.0
    cfg.normalization.enabled = True
    cfg.rep_segmentation.enabled = True      # preserves manual rep labels as manual_override
    cfg.phase_segmentation.enabled = False   # use recording-plane phase QC sample above for p01
    cfg.motion_attribution.enabled = False   # squat is bilateral_symmetric
    cfg.features.enabled = True
    cfg.biomech.enabled = True
    cfg.biomarker.enabled = True

    analysis_result_df, analysis_report = run_pipeline(
        analysis_df,
        cfg,
        ann_df=ann_df,
        landmarks=LANDMARKS,
    )

    pipeline_summary = {
        "validation_passed": analysis_report.get("validation", {}).get("passed"),
        "annotation_reps": analysis_report.get("annotation", {}).get("num_reps"),
        "rep_segmentation_status": analysis_report.get("rep_segmentation", {}).get("status"),
        "features": len(analysis_report.get("features", [])),
        "biomech": len(analysis_report.get("biomech", [])),
        "biomarkers": len(analysis_report.get("biomarker", [])),
        "scores": len(analysis_report.get("biomarker_scores", [])),
    }
    display(pd.DataFrame([pipeline_summary]))
    print("report keys:", sorted(analysis_report.keys()))

## Report Tables

These are research/provenance tables, not clinical diagnosis. The movement-quality score should be read together with validation, visibility, view reliability, preprocessing notes, and withheld/low-confidence feature metadata.

In [ ]:
if analysis_report is None:
    print("Skipped: run the pipeline after annotation is ready.")
else:
    score_df = pd.DataFrame(analysis_report.get("biomarker_scores", []))
    feature_df = pd.DataFrame(analysis_report.get("features", []))
    biomech_df = pd.DataFrame(analysis_report.get("biomech", []))

    display(Markdown("### Biomarker Scores"))
    if score_df.empty:
        print("No biomarker score records emitted.")
    else:
        display(score_df[["score_id", "exercise_id", "rep_id", "final_score", "domain_scores", "withheld_features"]])

    display(Markdown("### Feature Records"))
    if feature_df.empty:
        print("No feature records emitted.")
    else:
        display(
            feature_df[
                [
                    "feature_id",
                    "rep_id",
                    "value",
                    "unit",
                    "view_reliability",
                    "availability",
                    "availability_reasons",
                    "camera_zone",
                    "depth_dependency",
                    "model_depth_reliability",
                    "landmark_quality",
                ]
            ].sort_values(["rep_id", "feature_id"]).head(80)
        )

    display(Markdown("### Biomech Proxy Records"))
    if biomech_df.empty:
        print("No biomech records emitted.")
    else:
        display(
            biomech_df[
                [
                    "metric_id",
                    "rep_id",
                    "value",
                    "unit",
                    "visibility_weight_applied",
                    "n_frames_used",
                    "n_frames_excluded_low_visibility",
                    "note",
                ]
            ].sort_values(["rep_id", "metric_id"]).head(80)
        )

## Save Optional Outputs

Set `SAVE_OUTPUTS=True` in the input settings if you want the processed dataframe and JSON report written under `data/processed/reports/p01_squat_set1/`.

In [ ]:
if SAVE_OUTPUTS and analysis_report is not None and analysis_result_df is not None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    processed_path = OUTPUT_DIR / "p01_squat_set1_processed.csv"
    report_path = OUTPUT_DIR / "p01_squat_set1_report.json"
    analysis_result_df.to_csv(processed_path, index=False)
    with report_path.open("w", encoding="utf-8") as f:
        json.dump(analysis_report, f, indent=2, ensure_ascii=False, default=str)
    print("saved processed:", processed_path.relative_to(PROJECT_ROOT))
    print("saved report:", report_path.relative_to(PROJECT_ROOT))
else:
    print("SAVE_OUTPUTS is false or analysis report is not ready.")

## Pass Criteria

- Structural validation passes after empty-frame trim and missing-frame regularization.
- Annotation ranges are non-overlapping and within the original frame range.
- `rep_segmentation.status == manual_override` after pipeline run.
- Feature/biomech/biomarker tables are emitted per reviewed rep.
- Report text discloses side-view reliability limits and low-visibility landmarks.
- No clinical diagnosis or disease classification is inferred from these outputs.